[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IshtVibhu/Python-for-Computational-Economics/blob/main/NB_03_Pandas.ipynb)

# Python for Computational Economics
## Notebook 03 — Pandas: DataFrames for Economic Data

**Prerequisites:** NB_01, NB_02

**What you will learn:**
- Creating and inspecting DataFrames
- Selecting, filtering, and sorting data
- Merging datasets (analogous to database joins)
- GroupBy — computing statistics by group
- Time-series handling: resampling, rolling windows
- Storing and loading Sugarscape simulation output

In [ ]:
try:
    import pandas as pd
    import numpy as np
except ImportError:
    !pip install pandas numpy --quiet
    import pandas as pd
    import numpy as np

print(f"Pandas {pd.__version__}, NumPy {np.__version__}")

---
## 1. Creating DataFrames

A **DataFrame** is a table with labeled rows and columns. It is the central data structure for economic analysis in Python.

In [ ]:
# From a dictionary — most common way
macro_data = pd.DataFrame({
    "year":        [2018, 2019, 2020, 2021, 2022, 2023],
    "gdp_growth":  [2.9,  2.3, -3.4,  5.9,  2.1,  2.5],  # % real
    "inflation":   [2.4,  1.8,  1.2,  4.7,  8.0,  3.4],  # % CPI
    "unemployment":[3.9,  3.5,  8.1,  5.4,  3.6,  3.7],  # % of labour force
    "fed_rate":    [2.5,  1.75, 0.25, 0.25, 4.25, 5.25],  # %
})

print(macro_data)
print(f"\nShape: {macro_data.shape}")
print(f"\nData types:")
print(macro_data.dtypes)

In [ ]:
# Quick summary statistics — always run this first when exploring new data
macro_data.describe().round(2)

---
## 2. Selecting and Filtering Data

In [ ]:
# Select a single column (returns a Series)
inflation = macro_data["inflation"]
print("Inflation series:")
print(inflation.values)

# Select multiple columns
nominal_vars = macro_data[["year", "inflation", "fed_rate"]]
print("\nNominal variables:")
print(nominal_vars)

# Filter rows: years with recession or near-recession (growth < 0)
contractions = macro_data[macro_data["gdp_growth"] < 0]
print(f"\nContraction years:")
print(contractions)

# Compound filter: high inflation AND low unemployment (Phillips curve stress)
stress = macro_data[(macro_data["inflation"] > 4) & (macro_data["unemployment"] < 5)]
print(f"\nHigh inflation + low unemployment (stagflation risk):")
print(stress)

---
## 3. Adding Computed Columns

In [ ]:
df = macro_data.copy()

# Real interest rate = Fed rate - inflation (Fisher approximation)
df["real_rate"] = df["fed_rate"] - df["inflation"]

# Misery index = inflation + unemployment (Okun)
df["misery_index"] = df["inflation"] + df["unemployment"]

# Business cycle label
def classify_cycle(row):
    if row["gdp_growth"] < 0:
        return "Recession"
    elif row["gdp_growth"] < 1.5:
        return "Slow"
    elif row["gdp_growth"] < 3.5:
        return "Normal"
    else:
        return "Boom"

df["cycle_phase"] = df.apply(classify_cycle, axis=1)

print(df[["year", "gdp_growth", "real_rate", "misery_index", "cycle_phase"]])

---
## 4. GroupBy — Statistics by Group

`groupby` is the Pandas equivalent of SQL's `GROUP BY`. Essential for panel data and cross-sectional comparisons.

In [ ]:
np.random.seed(42)

# Simulate a cross-country panel dataset
countries = ["USA", "Germany", "Japan", "India", "Brazil"]
years     = list(range(2015, 2024))

panel = pd.DataFrame([
    {
        "country": c,
        "year":    y,
        "gdp_growth": np.random.normal(loc={"USA":2.5, "Germany":1.5, "Japan":1.0,
                                             "India":6.5, "Brazil":1.0}[c], scale=1.5),
        "inflation":  np.random.uniform(1, 8),
        "income_group": "High" if c in ["USA", "Germany", "Japan"] else "Middle",
    }
    for c in countries for y in years
])

# Average GDP growth by country
print("Average GDP growth by country:")
print(panel.groupby("country")["gdp_growth"].mean().round(2).sort_values(ascending=False))

# Multiple statistics by income group
print("\nBy income group:")
print(panel.groupby("income_group")[["gdp_growth", "inflation"]].agg(["mean", "std"]).round(2))

---
## 5. Merging Datasets

Economic analysis almost always requires joining multiple data sources. Pandas `merge` works like SQL JOINs.

In [ ]:
# Dataset 1: macro indicators
macro = pd.DataFrame({
    "country":    ["USA", "Germany", "Japan", "India"],
    "gdp_usd_bn": [26854, 4430, 4231, 3733],
    "population_m": [334, 84, 125, 1430],
})

# Dataset 2: development indicators
dev = pd.DataFrame({
    "country":      ["USA", "Germany", "India", "Brazil"],
    "gini":         [0.41, 0.29, 0.35, 0.49],
    "hdi":          [0.921, 0.950, 0.644, 0.760],
})

# Inner join — only countries in both datasets
inner = pd.merge(macro, dev, on="country", how="inner")
inner["gdp_per_capita"] = inner["gdp_usd_bn"] * 1e9 / (inner["population_m"] * 1e6)
print("Inner join (countries in both datasets):")
print(inner[["country", "gdp_per_capita", "gini", "hdi"]].round(0))

# Left join — keep all macro rows, fill missing dev data with NaN
left = pd.merge(macro, dev, on="country", how="left")
print("\nLeft join (all macro countries, NaN where dev data missing):")
print(left[["country", "gini", "hdi"]])

---
## 6. Time Series — Rolling Windows and Resampling

In [ ]:
# Monthly US CPI data (simulated around actual trend)
np.random.seed(0)
dates = pd.date_range("2018-01", "2024-01", freq="M")
cpi_monthly = pd.Series(
    100 + np.cumsum(np.random.normal(loc=0.25, scale=0.15, size=len(dates))),
    index=dates,
    name="CPI"
)

# Month-over-month inflation
mom_inflation = cpi_monthly.pct_change() * 100

# 12-month rolling average of MoM inflation ≈ year-on-year
rolling_annual = mom_inflation.rolling(window=12).mean() * 12

# Resample to quarterly
quarterly_cpi = cpi_monthly.resample("Q").mean()

# Combine into a DataFrame
ts_df = pd.DataFrame({
    "cpi":            cpi_monthly,
    "mom_inflation":  mom_inflation.round(3),
    "rolling_annual": rolling_annual.round(3),
})

print("Monthly CPI and inflation (last 12 months):")
print(ts_df.tail(12).round(2))
print(f"\nQuarterly CPI (last 4 quarters):")
print(quarterly_cpi.tail(4).round(2))

---
## 7. Storing Sugarscape Simulation Output

Simulation results are stored in a DataFrame — one row per agent per time step. This makes analysis with Pandas straightforward.

In [ ]:
np.random.seed(1)

# Simulate a minimal Sugarscape run (no actual movement — just for illustration)
N_AGENTS = 100
N_STEPS  = 20

records = []
wealth  = np.random.uniform(5, 30, size=N_AGENTS)

for step in range(N_STEPS):
    # Random multiplicative wealth change (trading)
    wealth *= np.random.lognormal(0, 0.1, size=N_AGENTS)
    wealth = np.clip(wealth, 0.1, None)  # no negative wealth

    for agent_id in range(N_AGENTS):
        records.append({
            "step":      step,
            "agent_id":  agent_id,
            "wealth":    round(wealth[agent_id], 2),
        })

sim_df = pd.DataFrame(records)

# Summary statistics per step
step_stats = sim_df.groupby("step")["wealth"].agg(
    mean_wealth="mean",
    median_wealth="median",
    max_wealth="max",
    std_wealth="std",
).round(2)

print("Sugarscape simulation — wealth statistics per step:")
print(step_stats.to_string())

---
## Your Turn — Gini Coefficient Over Time

Using the `sim_df` DataFrame from above, compute the **Gini coefficient** of wealth at each time step. Plot the result as a print statement showing step 0, 5, 10, 15, 19.

Hint: use `groupby("step")["wealth"].apply(gini_fn)` where `gini_fn` is a function that computes the Gini of a Series.

In [ ]:
# Your code here


In [ ]:
#@title Solution
def gini_series(s):
    w = s.values
    w_sorted = np.sort(w)
    n = len(w_sorted)
    ranks = np.arange(1, n + 1)
    return (2 * np.sum(ranks * w_sorted) / (n * w_sorted.sum())) - (n + 1) / n

gini_over_time = sim_df.groupby("step")["wealth"].apply(gini_series).round(3)
print("Gini coefficient over time:")
for step in [0, 5, 10, 15, 19]:
    print(f"  Step {step:2d}: Gini = {gini_over_time[step]:.3f}")
print("\nRising Gini reflects inequality growth from multiplicative shocks.")

---
## Summary

| Task | Pandas method |
|---|---|
| Create DataFrame | `pd.DataFrame({...})` |
| Select column | `df["col"]` |
| Filter rows | `df[df["col"] > val]` |
| Add computed column | `df["new"] = expression` |
| Group statistics | `df.groupby("col").agg(...)` |
| Join datasets | `pd.merge(df1, df2, on="key", how="inner")` |
| Time series | `pd.date_range()`, `.resample()`, `.rolling()` |

**Next:** NB_04_Matplotlib_Seaborn.ipynb — visualising economic data.